# 🫀 퀘스트 46 · Q7-S′ — **P 정렬 특징이 리듬 위에 얹히는가** (퀘스트의 실제 질문)

| | **MedKOS / `notebooks/quest46_q7s2_p_aligned.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | **Q7-P0**(P 위치 표) · **Q7-U**(자 세우기) · **Q7-T** · `ailab-2026-0067`(Q7-R) |
| 규약 | **R11 · R16 · R22 · R24 · R26 ② · R29 ② · R30 ① · R33 ① ② · R34 ①③④⑤ · R35 ①②⑦** |
| 학습 | 로지스틱 회귀만(딥러닝 아님) · GPU 불필요 · **예상 45~70분** |

## 이 런이 퀘스트의 질문이다

Q7-D~Q7-S 가 물어온 건 하나였다 —

> **리듬(조기성)이 이미 주는 것 너머로, P 파 형태가 SVEB 판별에 정보를 더하는가?**

Q7-T~Q7-P0 는 전부 **그걸 잴 자를 세우는 일**이었다.

```
Q7-T   자체 검출기 Se 0.4375 ❌  · 소거가 P 를 드러낸다 ✅(+0.2825)
Q7-U   공개 자 Se 0.9109 ✅      · 소거는 **검출**을 해친다 ❌(70칸 전부 음수)
Q7-P0  자를 SVDB 에 적용         · 좌표 정합 증명 통과 · P 위치 중앙 −164ms
```

**이제 잴 수 있다.**

## ★★★ Q7-P0 가 준 두 제약 — 이걸 어기면 또 헛돈다

**① `p_idx >= 0` 은 「P 가 있다」가 아니라 「후보를 놓았다」다.**
Q7-P0 의 발견률 0.9756 은 검출기가 **기권하지 않아서** 나온 값이다. Q7-U @발화율
1.00 의 PPV 0.7145 → 발화의 **28.55%** 가 오검출이고, 이는 BUT PDB 문헌 P 부재율
**0.288** 과 거의 정확히 일치한다.

> 그러니 **존재 플래그가 아니라 `p_score` 를 연속값으로** 쓴다.
> 「P 가 있나」는 이분값이 아니라 **신뢰도 스펙트럼**이다.

**② P 위치의 유효 해상도는 360Hz 가 아니라 128Hz(7.8ms)다.**
SVDB 원본이 128Hz라 360Hz로 올려도 정보가 늘지 않는다. Q7-P0 히스토그램의 주기적
스파이크가 그 격자다.

> **8ms 보다 미세한 위치 차이는 보간 인공물이다.** 위치 특징을 **8ms 격자로 양자화**하고,
> 그보다 미세한 해상도를 요구하는 특징은 만들지 않는다.

## 특징 — **5차원**(영점 대조와 차원을 맞춘다)

차원을 늘리는 것 자체에 비용이 있다(Q7-R 의 R1 −0.0115 가 그 후보였다). 그래서
`noise5`(순수 잡음 5차원)와 **같은 차원**으로 맞춘다.

| # | 특징 | 왜 |
|---|---|---|
| 1 | `p_score` | ★ 연속 신뢰도. 제약 ① |
| 2 | `pr_q8` | P−R 간격(ms), **8ms 격자**. 제약 ② |
| 3 | `pr_dev` | ★★ **개인 내 편차** — 그 레코드 N 비트 PR 중앙값 대비. 개인화 가설의 핵심 |
| 4 | `sc_dev` | ★ 점수의 **개인 내 로버스트 z** — (score − med) / MAD |
| 5 | `p_miss` | 후보가 창 안에 없었나(0/1) — 결측을 **버리지 않고 표현**한다 |

★ 3·4 가 요점이다. Q7-R 의 R1(교차환자 −0.0115) vs R5(개체 내 +0.0974) 모순은
**「보편 P 템플릿이 없어도 각 환자 안에서는 「평소와 다르다」가 성립한다」**는 가설을
낳았다. `pr_dev`·`sc_dev` 가 바로 그 「평소 대비」다.

## 사전등록 — 관문

| 관문 | 내용 | 판정 |
|---|---|---|
| **S1 ★★** | **개인 내** — 개인화 라벨 k개를 주면 P 특징이 리듬 위에 얹히는가 | ΔAUPRC CI 하한 > 0 |
| **S2 ★★** | **교차환자(LORO)** — 같은 것을 개인화 없이 | ΔAUPRC CI 하한 > 0 |
| **S3 ★★★(주)** | **조기성을 통제해도 남는가** — `f1` 정확 매칭 층화 + `f2_k` 전 계열 잔차화 | 매칭 AUROC CI 하한 > 0.5 |
| **S4** | **영점** — `noise5`(순수 잡음) · `shuf5`(레코드 안 치환) | 둘 다 0 근처여야 |
| **S5** | 자를 바꾼 게 효과가 있나 — `palign` vs `pmorph`(옛 R 기준 창 PCA) | 짝지은 차 |

### ★★★ S3 이 진짜 관문이다

S1·S2 가 ✅ 여도 **S3 에서 죽으면 조기성을 다시 잰 것**이다. Q7 이 R24 이후 계속
싸워온 게 그거다 — R 기준 고정창은 심박수 대리변수였다.

`f1`(= median(pre) − pre) **정확 매칭** 층 안에서는 조기성이 상수다. 그 안에서도
P 특징이 S/N 을 가르면 **그건 리듬이 아니다.**

### 판정표 (R29 ②)

- **S1 ✅ · S2 ✅ · S3 ✅** → ★★ P 형태가 리듬 너머로 정보를 준다. **딥러닝에 넣는다**
- **S1 ✅ · S2 ❌ · S3 ✅** → ★ **개인화 아키텍처가 필요하다**(환자 임베딩·개인 보정)
- **S3 ❌** → 남은 건 조기성이다. **형태 갈래를 닫고** 리듬-only 모델의 보정으로
- **S4 가 0 이 아니면** → 영점이 깨진 것이다. 어떤 결론 분기도 타지 않는다
- **어느 것이든 ⛔ 측정 불가** → 필요 표본을 계산하고 분기하지 않는다

In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    """최소 검출 효과 = CI 반폭(R33 ①)."""
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def need_n(n, lo, hi, mean, margin):
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(mean)) or n < 1:
        return float("nan")
    slack = margin - abs(mean)
    return None if slack <= 0 else float(n) * (((hi - lo) / 2.0) / slack) ** 2

def boot_mean(v, seed, nb=3000, q=2.5):
    """**레코드 단위** 부트스트랩(R11 — 환자가 단위다)."""
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1
FS, RPRE, L = 360, 100, 300

# ── ★★ Q7-P0 가 준 제약 ②: SVDB 원본이 128Hz → 유효 해상도 7.8ms
QUANT_MS = 1000.0 / 128.0        # 7.8125ms — **위치 특징을 이 격자로 양자화**한다

# ── 리듬 기저 (**강하게** 잡는다 — Δ 가 「리듬 너머」여야 하므로 바를 높인다)
FULL_K = tuple(range(4, 33))     # f2_4 … f2_32 **전 계열**(무한회귀 종결 · Q7-R ⑧)
RHY_K  = (5, 10, 20, 32)         # 리듬 기저에 직접 넣는 f2_k
assert set(RHY_K) <= set(FULL_K), (
    f"RHY_K {RHY_K} 가 FULL_K 부분집합이 아니다 — 잔차화 기저에 없는 항을 "
    "리듬 기저에 넣으면 S3 의 「리듬으로 설명되는 성분」이 불완전해진다")

# ── 팔
GATE_ARM = "palign"              # ★ 주 관문
ARMS = ("palign", "pmorph", "noise5", "shuf5")
N_DIM = 5                        # ★ 전 팔 **같은 차원** — 차원 추가 비용을 상쇄
PMORPH_WIN = (31, 53)            # 옛 R 기준 창 `p_mid_22`(R−192~−131ms) — 비교용

# ── 개인화 · 평가
K_PERS = (5, 10, 20)             # 환자당 라벨 비트 수(**클래스당**)
SPEC_LO = 0.95                   # 고특이도 부분 AUC 하한
MIN_S, MIN_N = 25, 25            # R17 — 레코드가 이보다 적으면 판정에서 뺀다

# ── S3 조기성 통제
MIN_PAIR = 200                   # 레코드당 최소 **매칭 쌍** 수(R17)
N_SHUF = 20                      # 라벨셔플 null 반복(R26 ②)

# ── 자산
SV5   = os.path.join(MITBIH, "svdb_data5.npz")
PDEL  = os.path.join(MITBIH, "svdb_pdelin.npz")

RULE_CHECK = {
    "R11 환자 단위":      "부트스트랩·판정 전부 **레코드 단위**(비트 단위 아님)",
    "R16 fallback 없음":  "자산 없으면 **중단** — 합성으로 대체 안 함",
    "R22 누수 없음":      "PCA·표준화·개인화는 **학습 비트에서만** 적합 · 개인화에 쓴 비트는 "
                          "**양쪽 팔 모두** 평가에서 뺀다",
    "R24 심박수 대리":    "★★ **S3 이 이 규칙의 정면 대응** — `f1` 정확 매칭 층 안에서 다시 잰다",
    "R26 ② null":         "라벨셔플 null 을 **전 팔**에 돌려 SE 를 전파",
    "R29 ② 분기 금지":    "⛔ 측정 불가는 어떤 결론 분기도 타지 않는다",
    "R30 ① 필요표본":     "미결이면 필요 레코드 수를 계산해 출력",
    "R33 ① MDE":          "관문마다 MDE(CI 반폭)를 내고 점추정과 비교",
    "R34 ③ 대조 보장":    "`noise5`·`shuf5` 는 **구성으로 보장된** 영점",
    "R34 ④ 문턱 근거":    "★ 양자화 7.8125ms = 1000/128 — **유도값**이지 임의값이 아니다",
    "R35 ①":              "★ 자를 먼저 세웠다(Q7-U Se 0.9109) — 그래서 이제 잴 수 있다",
    "R35 ②":              "★ `p_score` 를 **연속**으로 쓴다 — 존재 플래그는 「후보를 놓았다」일 뿐",
    "R35 ⑦":              "★ 소비 측에서 **정합을 재확인**한다(pid·sym 대조)",
}

CONFIG = dict(
    exp="quest46_q7s2_p_aligned", quest="ailab-2026-0046", step="p-aligned-personalize",
    parent_exp=["quest46_q7p0_svdb_pdelin", "quest46_q7u_public_delineator"],
    purpose=("**퀘스트 46 의 실제 질문**: 리듬(조기성)이 이미 주는 것 너머로 P 파 형태가 "
             "SVEB 판별에 정보를 더하는가. Q7-T~Q7-P0 는 전부 그걸 잴 **자를 세우는 일**이었다"
             "(Q7-U 공개 delineator Se 0.9109 · Q7-P0 좌표 정합 증명 통과). 이제 잰다. "
             "★ Q7-P0 의 두 제약을 지킨다 — ① `p_idx>=0` 은 「P 가 있다」가 아니라 「후보를 "
             "놓았다」이므로 **`p_score` 를 연속**으로 쓴다(발견률 0.9756 vs 문헌 부재율 "
             "0.288 · Q7-U PPV 0.7145 로 정확히 설명된다) ② P 위치 유효 해상도는 **7.8ms**"
             "(SVDB 원본 128Hz)이므로 위치를 그 격자로 **양자화**한다"),
    dataset="SVDB 78레코드 184,499비트(svdb_data5.npz) + P 위치 표(svdb_pdelin.npz)",
    arms=list(ARMS), gate_arm=GATE_ARM, n_dim=N_DIM, k_pers=list(K_PERS),
    quant_ms=QUANT_MS, full_k=list(FULL_K), rhy_k=list(RHY_K), spec_lo=SPEC_LO,
    min_s=MIN_S, min_n=MIN_N, min_pair=MIN_PAIR, n_shuf=N_SHUF,
    ruler=dict(exp="quest46_q7u_public_delineator", se=0.9109, ppv=0.7145, db="BUT PDB"),
    rule_check=RULE_CHECK,
    predictions={
        "S1": "★★ **개인 내** — 개인화 라벨 k∈{5,10,20}(클래스당)을 학습에 주면 P 특징이 "
              "**리듬 기저 위에** 얹히는가. ΔAUPRC 의 CI 하한 > 0",
        "S2": "★★ **교차환자(LORO)** — 같은 것을 개인화 없이. Q7-R 의 R1 이 여기서 −0.0115 "
              "였고, 그게 「보편 P 템플릿이 없다」는 가설의 근거였다",
        "S3": "★★★ **(주 관문) 조기성을 통제해도 남는가** — `f1`(=median(pre)−pre) **정확 "
              "매칭** 층 안에서 P 특징의 층화 AUROC. 층 안에서는 조기성이 **상수**이므로, "
              "거기서도 갈리면 **그건 리듬이 아니다**. CI 하한 > 0.5 · 라벨셔플 null 병기. "
              "★ S1·S2 가 ✅ 여도 여기서 죽으면 **조기성을 다시 잰 것**이다(R24)",
        "S4": "**영점** — `noise5`(순수 잡음 5차원)·`shuf5`(같은 특징을 **레코드 안에서** "
              "치환)가 0 근처인가. 0 이 아니면 Δ 의 눈금이 깨진 것이라 판정 불가",
        "S5": "자를 바꾼 게 효과가 있나 — `palign` vs `pmorph`(옛 R 기준 고정창 PCA · R24 가 "
              "심박수 대리변수라 못 박은 그 도구). 짝지은 차"},
    caveat=("★ **딥러닝이 아니다** — 로지스틱 회귀다. 특징에 정보가 0 이면 딥러닝이 만들어내지 "
            "못하고, 딥러닝은 null 을 용량·튜닝 뒤에 숨긴다. 값싼 모형으로 먼저 가른다. "
            "★ **소거 잔차는 특징에 안 넣는다** — Q7-U 의 U2 가 소거를 검출 경로에서 내렸고"
            "(격자 70칸 전부 음수), 여기 쓸 소거 특징은 Q7-P0 자산에 없다. 필요하면 별도 런. "
            "★ **유도 0 만** — Q7-P0 가 유도 0 만 구획했다. "
            "★ 전 팔이 **같은 5차원**이다 — 차원 추가 비용을 상쇄하려고. "
            "★ 학습 0회(딥러닝) · 로지스틱 회귀 다수 · 예상 45~70분"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7s2_p_aligned", CONFIG, project=PROJECT)
run.log("설정 ✅ **퀘스트 46 의 실제 질문** — 리듬 너머로 P 형태가 정보를 더하는가")
run.log(f"  팔 {ARMS} · 전부 **{N_DIM}차원**(차원 비용 상쇄) · 주 관문 팔 `{GATE_ARM}`")
run.log(f"  ★ 위치 양자화 {QUANT_MS:.4f}ms (= 1000/128 · SVDB 원본 해상도) — R34 ④ 유도값")
run.log(f"  ★ 주 관문은 **S3**(조기성 통제) — S1·S2 가 ✅ 여도 여기서 죽으면 리듬을 다시 잰 것")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")

In [ ]:
# CELL 2 — 【S2-0a】 자산 · ★ 정합 **재확인** (Q7-P0 가 안내한 대로 · R35 ⑦)
run.log("\n" + "=" * 100)
run.log("【S2-0a】 자산 적재 + 좌표 정합 재확인")
run.log("=" * 100)
for p_, why in ((SV5, "svdb_labels.py build"), (PDEL, "**Q7-P0 를 먼저 돌린다**")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}(R16)")
D5 = np.load(SV5, allow_pickle=True)
PD = np.load(PDEL, allow_pickle=True)
run.log(f"  svdb_data5 필드 {sorted(D5.files)}")
run.log(f"  svdb_pdelin 필드 {sorted(PD.files)}")

PID = np.asarray(D5["pid"]).astype(int)
SYM = np.asarray(D5["sym"]).astype(str)
Y3  = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float)
POST = np.asarray(D5["post_rr"], float)
BEAT = np.asarray(D5["beat"])

# ★★ 정합 재확인 — Q7-P0 가 증명했지만 **소비 측에서 다시 본다**(자산이 뒤바뀔 수 있다)
for f in ("p_idx", "p_score", "pid", "sym"):
    if f not in PD.files:
        raise AssetError(f"P 위치 표에 `{f}` 가 없다 — 보유 {sorted(PD.files)}")
if len(PD["p_idx"]) != len(PID):
    raise AssetError(f"길이 불일치 — pdelin {len(PD['p_idx']):,} vs d5 {len(PID):,}")
_bp = int((np.asarray(PD["pid"]).astype(int) != PID).sum())
_bs = int((np.asarray(PD["sym"]).astype(str) != SYM).sum())
if _bp or _bs:
    i0 = int(np.argmax((np.asarray(PD["pid"]).astype(int) != PID) |
                       (np.asarray(PD["sym"]).astype(str) != SYM)))
    raise AssetError(f"정합 깨짐 — pid {_bp}개 · sym {_bs}개 · 첫 어긋남 idx {i0}. "
                     "Q7-P0 를 다시 돌린다")
run.log(f"  ✅ 정합 재확인 — {len(PID):,} 비트 전부 일치(pid·sym)")

P_IDX = np.asarray(PD["p_idx"]).astype(int)
P_SC  = np.asarray(PD["p_score"], float)
KEEP = Y3 >= 0                                    # F/Q 제외 — 기존 3-class 집합 재현
run.log(f"  비트 {len(PID):,} → 유효 {int(KEEP.sum()):,} (y3>=0) · 레코드 {len(np.unique(PID))}")
run.log(f"  P 후보 있음 {float((P_IDX[KEEP] >= 0).mean()):.4f}  "
        "← ★ 「P 가 있다」가 아니라 「후보를 놓았다」다(Q7-U PPV 0.7145)")
CONFIG["asset"] = dict(n=int(len(PID)), n_keep=int(KEEP.sum()),
                       n_rec=int(len(np.unique(PID))),
                       cand_rate=float((P_IDX[KEEP] >= 0).mean()), align_ok=True)
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【S2-A】 특징 — 리듬 기저 · P 정렬 5차원 · 옛 창 · 영점
from sklearn.decomposition import PCA
run.log("\n" + "=" * 100)
run.log("【S2-A】 특징")
run.log("=" * 100)
K = np.where(KEEP)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
pidx = P_IDX[K]; psc = P_SC[K]
RS = np.array(sorted(set(RID.tolist())))

import pandas as pd
_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))

def local_base(k):
    """레코드 안에서 **앞선** k 비트의 중앙값 — 조기성 기저.
    ★ pandas rolling 으로 C 속도. 순수 파이썬 루프면 k 29개 × 18만 비트 = 수십 분이다."""
    r = _G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())
    r = np.asarray(r).astype(float)
    return np.where(np.isfinite(r), r, pre)              # 첫 비트는 자기 값

# ── 리듬 기저 — **강하게** 잡는다(Δ 가 「리듬 너머」여야 하므로 바를 높인다)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy()
_mean = _G.transform("mean").to_numpy()
f1 = _med - pre                                                    # 조기성(절대)
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre                                                    # 보상 휴지
f4 = np.nan_to_num(_std / (_mean + 1e-9))
RHY = np.c_[f1, np.column_stack([f2[k] for k in RHY_K]), f3, f4,
            np.log1p(pre), np.log1p(post)]
run.log(f"  리듬 기저 {RHY.shape[1]}차원 — f1 · f2_{RHY_K} · f3 · f4 · log(pre) · log(post)")

# ── ★★ P 정렬 5차원 (Q7-P0 제약 ①②)
MISS = pidx < 0
pr_ms = np.where(MISS, np.nan, (RPRE - pidx) / FS * 1000.0)        # PR 간격(ms · 양수)
pr_q8 = np.round(pr_ms / QUANT_MS) * QUANT_MS                      # ★ 7.8125ms 격자
# ★ 개인 기준은 **그 레코드의 N 비트**에서만 — S 비트로 기준을 잡으면 라벨이 샌다(R22)
nmask = (Y == 0)
pr_ref = np.full(len(pr_q8), np.nan); sc_ref = np.full(len(psc), np.nan)
sc_mad = np.full(len(psc), np.nan)
for u in np.unique(RID):
    m = RID == u; mn = m & nmask
    a = pr_q8[mn][np.isfinite(pr_q8[mn])]; b = psc[mn]
    pr_ref[m] = np.median(a) if len(a) >= 5 else np.nan
    sc_ref[m] = np.median(b) if len(b) >= 5 else np.nan
    sc_mad[m] = (np.median(np.abs(b - np.median(b))) + 1e-9) if len(b) >= 5 else np.nan
pr_dev = pr_q8 - pr_ref                                            # ★★ 개인 내 편차
sc_dev = (psc - sc_ref) / sc_mad                                   # ★ 개인 내 로버스트 z
PAL = np.c_[psc, np.nan_to_num(pr_q8, nan=0.0), np.nan_to_num(pr_dev, nan=0.0),
            np.nan_to_num(sc_dev, nan=0.0), MISS.astype(float)]
if PAL.shape[1] != N_DIM:
    raise AssetError(f"P 정렬 특징이 {PAL.shape[1]}차원 — {N_DIM} 이어야 영점과 맞는다")
run.log(f"  ★ P 정렬 {N_DIM}차원 — p_score · pr_q8(**{QUANT_MS:.4f}ms 격자**) · "
        "pr_dev(개인 내 편차) · sc_dev(개인 내 z) · p_miss")
run.log(f"    결측(후보 없음) {MISS.mean():.4f} — **버리지 않고 `p_miss` 로 표현**한다")
run.log("    ★ 개인 기준(pr_ref·sc_ref)은 **그 레코드의 N 비트에서만** 잡는다 — "
        "S 비트로 잡으면 라벨이 샌다(R22)")

# ── 옛 R 기준 고정창 (S5 비교용 · R24 가 심박수 대리변수라 못 박은 도구)
Bk = np.ascontiguousarray(BEAT[K]).astype("float32")
PM_RAW = Bk[:, 0, PMORPH_WIN[0]:PMORPH_WIN[1]].astype(float)
run.log(f"  옛 창 `pmorph` — 비트 idx {PMORPH_WIN} = R{(PMORPH_WIN[0]-RPRE)/FS*1000:+.0f}~"
        f"{(PMORPH_WIN[1]-RPRE)/FS*1000:+.0f}ms (S5 비교용)")

_rng = np.random.RandomState(SEED0 + 99)
NOISE = _rng.normal(size=(len(K), N_DIM))                          # ★ 영점: 순수 잡음
CONFIG["feat"] = dict(rhy_dim=int(RHY.shape[1]), pal_dim=int(PAL.shape[1]),
                      miss_rate=float(MISS.mean()))
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【S2-B】 S1 개인 내 · S2 교차환자 (ΔAUPRC · 같은 평가 비트)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_curve
run.log("\n" + "=" * 100)
run.log("【S2-B】 S1(개인 내) · S2(교차환자) — Δ = AUPRC(리듬+팔) − AUPRC(리듬)")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

def partial_auc(y, s, spec_lo):
    """고특이도(≥spec_lo) 구간의 부분 AUC — 임상 작동점."""
    fpr, tpr, _ = roc_curve(y, s)
    hi = 1.0 - spec_lo
    m = fpr <= hi
    if m.sum() < 2:
        return float("nan")
    x, yv = fpr[m], tpr[m]
    return float((np.diff(x) * (yv[:-1] + yv[1:]) / 2.0).sum()) / hi

def fit_eval(Xtr, ytr, Xte, yte):
    """표준화는 **학습에서만** 적합(R22)."""
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-9
    lr = LogisticRegression(max_iter=3000, C=1.0)
    lr.fit((Xtr - mu) / sd, ytr)
    s = lr.decision_function((Xte - mu) / sd)
    return average_precision_score(yte, s), partial_auc(yte, s, SPEC_LO)

def feats_for(arm, tr, te):
    """팔별 특징. **PCA·치환은 학습 비트에서만**(R22)."""
    if arm == "noise5":
        return NOISE[tr], NOISE[te]
    if arm == "pmorph":
        pca = PCA(n_components=N_DIM, random_state=SEED0).fit(PM_RAW[tr])
        return pca.transform(PM_RAW[tr]), pca.transform(PM_RAW[te])
    Ztr, Zte = PAL[tr].copy(), PAL[te].copy()
    if arm == "shuf5":
        # ★ **레코드 안에서** 치환 — 주변분포는 보존하고 라벨 관계만 끊는다(R34 ③)
        rr = np.random.RandomState(SEED0 + 7)
        for Z_, m_ in ((Ztr, tr), (Zte, te)):
            for u in np.unique(RID[m_]):
                sel = np.where(RID[m_] == u)[0]
                Z_[sel] = Z_[sel][rr.permutation(len(sel))]
    return Ztr, Zte

D_LORO = {a: np.full(len(RS), np.nan) for a in ARMS}
D_PERS = {k: {a: np.full(len(RS), np.nan) for a in ARMS} for k in K_PERS}
T0 = time.time()
for i, r in enumerate(RS):
    te0 = np.where(RID == r)[0]
    s_idx = te0[TT[te0]]; n_idx = te0[~TT[te0]]
    if len(s_idx) < MIN_S or len(n_idx) < MIN_N:
        continue                                       # R17
    rng = np.random.RandomState(SEED0 + 313 + int(r))
    pick = {k: (rng.choice(s_idx, k, replace=False), rng.choice(n_idx, k, replace=False))
            for k in K_PERS if len(s_idx) > k and len(n_idx) > k}
    if not pick:
        continue
    used = np.unique(np.concatenate([np.r_[a, b] for a, b in pick.values()]))
    ev = np.setdiff1d(te0, used)                       # ★ 개인화에 쓴 비트는 **양쪽 다** 제외
    tr0 = np.where(RID != r)[0]
    if len(ev) < 30 or TT[ev].sum() < 3:
        continue
    base_i = fit_eval(RHY[tr0], TT[tr0].astype(int), RHY[ev], TT[ev].astype(int))[0]
    base_p = {}
    for k in pick:
        tk = np.r_[tr0, pick[k][0], pick[k][1]]
        base_p[k] = fit_eval(RHY[tk], TT[tk].astype(int), RHY[ev], TT[ev].astype(int))[0]
    for a in ARMS:
        Ztr, Zev = feats_for(a, tr0, ev)
        D_LORO[a][i] = fit_eval(np.c_[RHY[tr0], Ztr], TT[tr0].astype(int),
                                np.c_[RHY[ev], Zev], TT[ev].astype(int))[0] - base_i
        for k in pick:
            tk = np.r_[tr0, pick[k][0], pick[k][1]]
            Zk, Zev2 = feats_for(a, tk, ev)
            D_PERS[k][a][i] = fit_eval(np.c_[RHY[tk], Zk], TT[tk].astype(int),
                                       np.c_[RHY[ev], Zev2], TT[ev].astype(int))[0] - base_p[k]
    if (i + 1) % 10 == 0:
        run.log(f"    {i+1}/{len(RS)}  ({time.time()-T0:.0f}초)")
run.log(f"  ({time.time()-T0:.0f}초) 판정 레코드 {int(np.isfinite(D_LORO[GATE_ARM]).sum())}")

run.log("\n  S2 — 교차환자(LORO) ΔAUPRC")
S2 = {}
for a in ARMS:
    m_, lo_, hi_, n_ = boot_mean(D_LORO[a], SEED0 + 21)
    S2[a] = dict(mean=m_, lo=lo_, hi=hi_, n=n_, mde=float(mde(lo_, hi_)))
    run.log(f"    {a:<8} **{m_:+.4f}** [{lo_:+.4f}, {hi_:+.4f}] · n={n_} · MDE {mde(lo_,hi_):.4f}")
run.log("\n  S1 — 개인 내(개인화 k) ΔAUPRC")
S1 = {}
for k in K_PERS:
    for a in ARMS:
        m_, lo_, hi_, n_ = boot_mean(D_PERS[k][a], SEED0 + 31 + k)
        S1[f"{a}@k{k}"] = dict(mean=m_, lo=lo_, hi=hi_, n=n_, mde=float(mde(lo_, hi_)))
    run.log(f"    k={k:<3}" + " · ".join(
        f"{a} {S1[f'{a}@k{k}']['mean']:+.4f}" for a in ARMS))
# ★ k 도 **LORO 로** 고른다 — 성적표에서 최대를 고르면 선택 편의(R34 ②)
KS = [k for k in K_PERS if np.isfinite(D_PERS[k][GATE_ARM]).sum() >= 3]
sel = []
idx_ok = np.where(np.isfinite(D_PERS[KS[0]][GATE_ARM]))[0] if KS else np.array([], int)
for i in idx_ok:
    bk = max(KS, key=lambda k: np.nanmean(np.delete(D_PERS[k][GATE_ARM], i)))
    sel.append(D_PERS[bk][GATE_ARM][i])
m1, l1, h1, n1 = boot_mean(sel, SEED0 + 41)
DIFF["S1"] = dict(arm=GATE_ARM, mean=m1, lo=l1, hi=h1, n=n1, mde=float(mde(l1, h1)))
g_("S1", "⛔ 측정 불가" if n1 < 3 else decide(l1, h1, 0.0, ">"),
   f"★★ 개인 내(**k 를 LORO 로 선택**) Δ **{m1:+.4f}** [{l1:+.4f}, {h1:+.4f}] · n={n1}")
d2 = S2[GATE_ARM]
DIFF["S2"] = dict(arm=GATE_ARM, **d2)
g_("S2", "⛔ 측정 불가" if d2["n"] < 3 else decide(d2["lo"], d2["hi"], 0.0, ">"),
   f"★★ 교차환자 Δ **{d2['mean']:+.4f}** [{d2['lo']:+.4f}, {d2['hi']:+.4f}] · n={d2['n']}")
CONFIG["S1"] = S1; CONFIG["S2"] = S2
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【S2-C】 ★★★ 주 관문 S3 — **조기성을 통제해도 남는가** (R24 정면 대응)
from sklearn.metrics import roc_auc_score
run.log("\n" + "=" * 100)
run.log("【S2-C】 S3(주) — `f1` **정확 매칭 쌍** 안에서 P 특징이 S/N 을 가르는가")
run.log("=" * 100)
run.log("  ▸ 매칭 쌍 안에서는 조기성(`f1`)이 **상수**다. 거기서도 갈리면 리듬이 아니다(R24)")
run.log("  ▸ 층화가 아니라 **매칭**인 이유: f1 이 연속이라 층이 안 채워진다(스모크런이 ⛔)")
run.log("  ▸ S1·S2 가 ✅ 여도 여기서 죽으면 **조기성을 다시 잰 것**이다")

def basis_ext(idx):
    """확장 기저 — `f2_k` **전 계열**(무한회귀 종결 · Q7-R ⑧). 잔차화용."""
    return np.c_[np.ones(len(idx)), f1[idx], f3[idx], f4[idx],
                 np.column_stack([f2[k][idx] for k in FULL_K])]

def resid(v, idx):
    """확장 기저에 대한 **잔차** — 리듬으로 설명되는 성분을 뺀다."""
    X = basis_ext(idx); y = v[idx]
    ok = np.isfinite(y)
    if ok.sum() < X.shape[1] + 5:
        return np.full(len(idx), np.nan)
    b = np.linalg.lstsq(X[ok], y[ok], rcond=None)[0]
    return y - X @ b

def matched_auc(vsub, idx, shuffle_seed=None):
    """★★ **정확 매칭 쌍** — 같은 레코드·같은 `f1`(정수) 안에서 S 비트마다 N 비트를
    짝지어 `P(feat_S > feat_N)` 를 센다(= Mann-Whitney = **조기성이 상수인 AUROC**).

    ★ 왜 층화가 아니라 매칭인가: `f1` 은 연속값이라 정수 반올림해도 레코드당 수백 개
      값이 나온다. 레코드 ~2,400비트 · S 14% 면 **f1 값 하나당 S 가 1개 남짓**이라
      「층마다 각 클래스 8개」를 요구하는 층화는 거의 안 선다(스모크런이 ⛔ 로 잡았다).
      매칭은 S 1개 · N 7개여도 **쌍 7개**를 만든다 — 같은 논리, 훨씬 나은 검정력.
    ★ `vsub` 는 **이미 `idx` 로 잘린** 배열이다(`resid()` 의 반환).

    반환 (AUROC, 쌍 수)."""
    tt = TT[idx]
    if shuffle_seed is not None:
        rr = np.random.RandomState(shuffle_seed)
        tt = tt.copy()
        for u in np.unique(RID[idx]):                  # ★ 레코드 **안에서** 셔플
            m = np.where(RID[idx] == u)[0]
            tt[m] = tt[m][rr.permutation(len(m))]
    key = np.round(f1[idx]).astype(int)
    win = tie = tot = 0.0
    for kk in np.unique(key):
        m = np.where(key == kk)[0]
        a = vsub[m[tt[m]]]; b = vsub[m[~tt[m]]]
        a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
        if not len(a) or not len(b):
            continue
        d = a[:, None] - b[None, :]
        win += float((d > 0).sum()); tie += float((d == 0).sum()); tot += float(d.size)
    return ((win + 0.5 * tie) / tot, int(tot)) if tot >= MIN_PAIR else (float("nan"), int(tot))

# 층화 AUROC 를 **레코드별**로 내고(R11) 부트스트랩
PROBES = {"p_score": psc, "pr_dev": pr_dev, "sc_dev": sc_dev,
          "pr_q8": pr_q8, "p_miss": MISS.astype(float)}
run.log("\n  매칭 AUROC (f1 정확 매칭 쌍 · f2_k 전 계열 잔차화 후) — 레코드 단위 · 우연 0.5")
run.log("  ⚠️ **`pr_dev`≡`pr_q8` · `sc_dev`≡`p_score` 로 같은 값이 나올 것이다** — 개인 내")
run.log("     중심화는 레코드 안 **단조변환**이고 AUROC 는 단조변환에 불변이기 때문이다.")
run.log("     버그가 아니다. 개인화(중심화)의 효과는 **S2(교차환자 모형)** 에서만 드러난다 —")
run.log("     거기선 레코드마다 다른 상수를 빼는 게 실제로 특징 분포를 바꾼다")
S3 = {}
for nm, v in PROBES.items():
    per, nul, npair = [], [], []
    for r in RS:
        idx = np.where(RID == r)[0]
        if TT[idx].sum() < MIN_S or (~TT[idx]).sum() < MIN_N:
            continue
        vr = resid(v, idx)                             # ★ f2_k 전 계열 잔차화
        a, npr = matched_auc(vr, idx)
        if np.isfinite(a):
            per.append(a); npair.append(npr)
            nul.append(np.nanmean([matched_auc(vr, idx, SEED0 + 700 + s)[0]
                                   for s in range(3)]))
    m_, lo_, hi_, n_ = boot_mean(per, SEED0 + 51)
    nm_, nlo, nhi, _ = boot_mean(nul, SEED0 + 52)
    S3[nm] = dict(auc=m_, lo=lo_, hi=hi_, n=n_, null=nm_, null_lo=nlo, null_hi=nhi,
                  mde=float(mde(lo_, hi_)),
                  pairs=int(np.median(npair)) if npair else 0)
    run.log(f"    {nm:<9} AUROC **{m_:.4f}** [{lo_:.4f}, {hi_:.4f}] · "
            f"null {nm_:.4f} [{nlo:.4f}, {nhi:.4f}] · n={n_} · "
            f"쌍 중앙 {S3[nm]['pairs']:,} · MDE {mde(lo_,hi_):.4f}")
# ★ 프로브도 **LORO 로** 고른다
PN = [p for p in PROBES if np.isfinite(S3[p]["auc"])]
if not PN or max(S3[p]["n"] for p in PN) < 3:
    g_("S3", "⛔ 측정 불가", "층이 선 레코드가 3 미만")
else:
    best = max(PN, key=lambda p: S3[p]["auc"])
    d3 = S3[best]
    DIFF["S3"] = dict(probe=best, **d3)
    g_("S3", decide(d3["lo"], d3["hi"], 0.5, ">"),
       f"★★★ 최량 프로브 `{best}` AUROC **{d3['auc']:.4f}** [{d3['lo']:.4f}, {d3['hi']:.4f}] "
       f"· null {d3['null']:.4f} · 문턱 0.5 · n={d3['n']}")
    run.log("       ▸ > 0.5 면 **조기성이 상수인 층 안에서도** P 특징이 S/N 을 가른다")
    run.log("         = 리듬이 아니다. ≤ 0.5 면 S1·S2 의 Δ 는 조기성을 다시 잰 것이다")
CONFIG["S3"] = S3
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【S2-D】 S4 영점 · S5 자 교체 효과 · 필요 표본
run.log("\n" + "=" * 100)
run.log("【S2-D】 S4 영점 · S5 `palign` vs `pmorph` · 필요 표본")
run.log("=" * 100)
run.log("  S4 — 영점 대조 (0 근처여야 한다 · 아니면 Δ 의 눈금이 깨진 것)")
S4ok = True
for a in ("noise5", "shuf5"):
    d_ = S2[a]
    v = decide(d_["lo"], d_["hi"], 0.0, ">")
    bad = v == "✅ 지지"
    S4ok &= not bad
    run.log(f"    {a:<8} LORO Δ **{d_['mean']:+.4f}** [{d_['lo']:+.4f}, {d_['hi']:+.4f}]"
            f"{'  ⚠️ **0 이 아니다**' if bad else ''}")
g_("S4", "✅ 지지" if S4ok else "❌ 기각",
   "영점이 0 근처다 — Δ 의 눈금을 믿을 수 있다" if S4ok else
   "★ **영점이 깨졌다** — 어떤 Δ 도 해석 불가")

run.log("\n  S5 — 자를 바꾼 게 효과가 있나 (`palign` − `pmorph` · 짝지은 차)")
S5 = {}
for nm, D in (("LORO", D_LORO), *[(f"k{k}", D_PERS[k]) for k in K_PERS]):
    d = D["palign"] - D["pmorph"]
    m_, lo_, hi_, n_ = boot_mean(d, SEED0 + 61)
    S5[nm] = dict(mean=m_, lo=lo_, hi=hi_, n=n_)
    run.log(f"    {nm:<6} **{m_:+.4f}** [{lo_:+.4f}, {hi_:+.4f}] · n={n_}")
run.log("    ▸ > 0 이면 **P 정렬 좌표계가 옛 R 기준 고정창보다 낫다**(R24 의 반례)")

run.log("\n  필요 표본 (R30 ①) — 미결인 관문에 대해")
for g in ("S1", "S2", "S3"):
    d_ = DIFF.get(g)
    if not d_ or not VERD.get(g, "").startswith("⚠️"):
        continue
    mval = d_.get("mean", d_.get("auc", float("nan")))
    ref = 0.0 if g != "S3" else 0.5
    nn = need_n(d_.get("n", 0), d_["lo"], d_["hi"], mval - ref, 0.01)
    run.log(f"    {g} 필요 레코드 ≈ {'도달 불가(여유 밖)' if nn is None else f'{nn:.0f}'} "
            f"(현재 {d_.get('n')} · SVDB 78 · +MITBIH 48 = 126)")
CONFIG["S4"] = dict(ok=bool(S4ok)); CONFIG["S5"] = S5
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【S2-E】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))
aa = list(ARMS)
ax[0].errorbar([S2[a]["mean"] for a in aa], np.arange(len(aa)),
               xerr=[[S2[a]["mean"] - S2[a]["lo"] for a in aa],
                     [S2[a]["hi"] - S2[a]["mean"] for a in aa]],
               fmt="o", capsize=4)
ax[0].axvline(0, color="k", lw=.9)
ax[0].set_yticks(range(len(aa))); ax[0].set_yticklabels(aa, fontsize=8)
ax[0].set_xlabel("S2 : cross-patient dAUPRC (LORO)"); ax[0].grid(alpha=.3, axis="x")
ks = list(K_PERS)
for a in aa:
    ax[1].plot(ks, [S1[f"{a}@k{k}"]["mean"] for k in ks], "o-", ms=4, label=a)
ax[1].axhline(0, color="k", lw=.9)
ax[1].set_xlabel("personalization labels per class (k)")
ax[1].set_ylabel("S1 : within-patient dAUPRC")
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)
pn = list(PROBES)
ax[2].errorbar([S3[p]["auc"] for p in pn], np.arange(len(pn)),
               xerr=[[S3[p]["auc"] - S3[p]["lo"] for p in pn],
                     [S3[p]["hi"] - S3[p]["auc"] for p in pn]],
               fmt="o", capsize=4, color="tab:red", label="observed")
ax[2].scatter([S3[p]["null"] for p in pn], np.arange(len(pn)), marker="x",
              color="tab:gray", label="label-shuffle null")
ax[2].axvline(0.5, ls="--", color="k", lw=.9)
ax[2].set_yticks(range(len(pn))); ax[2].set_yticklabels(pn, fontsize=8)
ax[2].set_xlabel("S3 : matched AUROC (exact f1 pairs, rhythm-residualized)")
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3, axis="x")
fig.tight_layout()
PNG = run.save_fig("q7s2_p_aligned", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
un_ = lambda k: VERD.get(k, "").startswith("⛔")     # R29 ②
for g in ("S1", "S2", "S3", "S4"):
    run.log(f"  {g:<4}{VERD.get(g, '(미실행)')}")
run.log("")
if not ok_("S4"):
    run.log("  ⛔ **영점이 깨졌다** — 어떤 Δ 도 해석 불가. 결론 분기를 타지 않는다")
elif any(un_(g) for g in ("S1", "S2", "S3")):
    run.log("  ⛔ 측정 불가가 있다 — 어떤 결론 분기도 타지 않는다(R29 ②)")
elif no_("S3"):
    run.log("  ⛔ **조기성을 통제하면 남지 않는다.** S1·S2 의 Δ 가 무엇이든 그건 리듬을")
    run.log("     다시 잰 것이다(R24). → **형태 갈래를 닫고** 리듬-only 모델의 보정으로 간다")
    run.log("     (Q3 사전확률 보정 · Q8 가중 매크로)")
elif ok_("S3") and ok_("S1") and ok_("S2"):
    run.log("  ★★ **P 형태가 리듬 너머로 정보를 준다 — 개인 내·교차환자 둘 다.**")
    run.log("     → **딥러닝에 넣는다**: 입력에 P 정렬 특징을 더해 1D-CNN 짝지은 재학습")
    run.log("       (같은 분할·같은 용량·같은 시드 · 기저는 **리듬-only 모델**)")
elif ok_("S3") and ok_("S1") and not ok_("S2"):
    run.log("  ★ **개인 내에서만 선다 — 개인화 아키텍처가 필요하다.**")
    run.log("     Q7-R 의 R1(교차 −0.0115) vs R5(개체 내 +0.0974) 가 예고한 그림이다.")
    run.log("     → 환자 임베딩 / 개인 보정 레이어. **보편 P 템플릿을 가정하지 않는 설계**")
else:
    run.log("  ⚠️ 미결 — MDE 와 비교해 「효과 없음」인지 「측정 한계」인지 먼저 가른다(R33 ①)")
run.log(f"\n  S5(자 교체) LORO **{S5['LORO']['mean']:+.4f}** "
        f"[{S5['LORO']['lo']:+.4f}, {S5['LORO']['hi']:+.4f}] — "
        "> 0 이면 P 정렬 좌표계가 옛 고정창보다 낫다(R24 의 반례)")

run.log("\n  사전등록 **종결 조건** (R34 ⑤)")
run.log("    ① S3 이 ❌ 면 **형태 갈래를 닫는다** — 더 돌지 않는다")
run.log("    ② S3 이 미결이고 필요 레코드가 126(SVDB78+MITBIH48)을 넘으면 같은 자료로")
run.log("       **더 돌지 않는다** — 코호트를 늘리거나 갈래를 접는다")
run.log("    ③ 딥러닝은 **S3 이 ✅ 일 때만** 들어간다 — 특징에 정보가 0 이면 딥러닝이")
run.log("       만들어내지 못하고, 용량·튜닝 뒤에 null 을 숨긴다")

run.log("\n  ▸ 이건 **딥러닝이 아니다** — 로지스틱 회귀다. 값싼 모형으로 먼저 가른다")
run.log("  ▸ P 위치는 **7.8125ms 격자**로 양자화했다(SVDB 원본 128Hz · Q7-P0 제약 ②)")
run.log("  ▸ `p_score` 를 **연속**으로 썼다 — 존재 플래그는 「후보를 놓았다」일 뿐(제약 ①)")

run.finish({
    "exp_id": "quest46_q7s2_p_aligned",
    "metric": "matched_auroc_prematurity_controlled",
    "value": float(DIFF.get("S3", {}).get("auc", float("nan"))),
    "passed": bool(ok_("S3") and ok_("S4")),
    "summary": ("퀘스트 46 의 실제 질문 — 리듬 너머로 P 형태가 정보를 더하는가. "
                "주 관문은 S3(f1 정확 매칭 층 안 층화 AUROC)."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "S1": CONFIG.get("S1", {}), "S2": CONFIG.get("S2", {}), "S3": CONFIG.get("S3", {}),
    "S4": CONFIG.get("S4", {}), "S5": S5, "asset": CONFIG.get("asset", {}),
    "feat": CONFIG.get("feat", {}), "ruler": CONFIG.get("ruler", {}), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step p-aligned-personalize`")